# QueryChat Response Style Experiment for AI Burnout Checkup Dashboard

This notebook evaluates potential user-facing controls for customizing the behavior of the QueryChat interface in the **AI Explorer** tab of the burnout_checkup dashboard.

## Objective 
This experiment evaluates whether response‑style control can serve as an intuitive and meaningful way for dashboard users to influence LLM outputs. The goal is to test whether different response styles produce noticeably different model responses and whether those differences enhance the overall usability of the AI Explorer.

The selected control will be integrated into the application and documented in the project specification.

## Control Selection

We considered several ways to customize QueryChat responses, including a verbosity slider, response-style selection, and scope restriction.

We finally selected a **Response Style control** because it produces meaningful differences in how the AI explains dataset insights while remaining simple and intuitive for users.

## Experiment: Response Style Comparison


### Experimental Setup

We generated responses to a consistent set of representative user questions using three response styles:

- Executive Summary
- Analytical Explanation
- Technical Interpretation

In [1]:
# Imports
import pandas as pd
import numpy as np
import os
import re

from pathlib import Path
from dotenv import load_dotenv
from chatlas import ChatAnthropic

In [2]:
# Load, merge, and view data
BASE_DIR = Path.cwd().parent
FEATURES_PATH = BASE_DIR / "data" / "raw" / "ai_productivity_features.csv"
TARGETS_PATH = BASE_DIR / "data" / "raw" / "ai_productivity_targets.csv"

features = pd.read_csv(FEATURES_PATH)
targets = pd.read_csv(TARGETS_PATH)

df = features.merge(targets, on="Employee_ID", how="left")
print("Table 1: Dataset Snapshot\n")
df.head()

Table 1: Dataset Snapshot



,Employee_ID,job_role,experience_years,ai_tool_usage_hours_per_week,tasks_automated_percent,manual_work_hours_per_week,learning_time_hours_per_week,deadline_pressure_level,meeting_hours_per_week,collaboration_hours_per_week,error_rate_percent,task_complexity_score,focus_hours_per_day,work_life_balance_score,burnout_risk_score,productivity_score,burnout_risk_level
0,3c6ca882-3fa3-446b-8208-c92f3f306f06,Writer,19,11.8,28.5,19.2,1.4,High,1.9,2.3,0.20,2,7.1,4.8,10.00,81.0,High
1,02f168cc-7747-4dbd-a868-ea2cfb41e22a,Designer,4,10.8,24.1,23.3,2.6,Low,8.0,9.8,1.82,3,3.4,5.5,6.78,59.2,Medium
2,d39ce8c9-6e2a-4f86-b888-e2b5f4a18cf7,Developer,6,25.9,69.4,10.0,1.4,Medium,6.8,8.9,5.52,5,4.6,3.8,9.66,62.4,High
3,14511660-d78a-453f-9449-f17cd239ec27,Manager,20,7.9,17.2,25.1,0.2,High,3.5,8.6,1.14,5,5.6,3.9,10.00,76.8,High
4,0597f0bb-ed5a-4e35-94ac-3f0f6a5c2bc2,Developer,15,8.6,20.6,20.1,1.4,Low,5.9,5.3,2.75,10,1.0,7.4,5.38,53.7,Medium


In [3]:
# Define the dataset description

data_description = """
    Employee-level workplace wellbeing and productivity dataset.
    
    Each row represents one employee.
    
    Columns:
    - Employee_ID: unique identifier for each employee.
    - job_role: employee job role/category.
    - experience_years: years of experience.
    - ai_tool_usage_hours_per_week: hours per week spent using AI tools.
    - tasks_automated_percent: percent of tasks automated with AI/tools.
    - manual_work_hours_per_week: hours per week spent on manual work.
    - learning_time_hours_per_week: hours per week spent learning new tools/workflows.
    - deadline_pressure_level: categorical deadline pressure level (Low, Medium, High).
    - meeting_hours_per_week: hours per week spent in meetings.
    - collaboration_hours_per_week: hours per week spent collaborating.
    - error_rate_percent: percentage error rate.
    - task_complexity_score: task complexity score.
    - focus_hours_per_day: average focus/deep work hours per day.
    - work_life_balance_score: numeric work-life balance score.
    - burnout_risk_score: numeric burnout risk score.
    - productivity_score: numeric productivity score.
    - burnout_risk_level: burnout category label.
    
    Use this dataset to analyze:
    - burnout risk by role
    - relationships between AI usage and burnout
    - productivity vs burnout
    - work-life balance patterns
    - workload-related drivers of burnout
    
    Important:
    - Stay grounded in the available dataset columns.
    - Do not make causal claims.
    - Frame findings as associations, patterns, or comparisons.
"""

In [4]:
# Create a compact dataset summary for the prompt
def build_dataset_snapshot(df: pd.DataFrame) -> str:
    role_counts = df["job_role"].value_counts().to_dict()
    burnout_counts = df["burnout_risk_level"].value_counts().to_dict()

    summary = {
        "n_rows": int(len(df)),
        "job_roles": role_counts,
        "burnout_levels": burnout_counts,
        "avg_ai_usage_hours": round(
            df["ai_tool_usage_hours_per_week"].mean(),
            2
        ),
        "avg_manual_hours": round(
            df["manual_work_hours_per_week"].mean(), 2),
        "avg_burnout_score": round(
            df["burnout_risk_score"].mean(), 
            2
        ),
        "avg_productivity_score": round(df["productivity_score"].mean(), 2),
        "avg_work_life_balance": round(
            df["work_life_balance_score"].mean(),
            2
        ),
    }

    lines = ["Dataset snapshot:"]
    for k, v in summary.items():
        lines.append(f"- {k}: {v}")
    return "\n".join(lines)

dataset_snapshot = build_dataset_snapshot(df)
print(dataset_snapshot)

Dataset snapshot:
- n_rows: 4500
- job_roles: {'Developer': 1115, 'Analyst': 892, 'Designer': 722, 'Marketer': 658, 'Manager': 652, 'Writer': 461}
- burnout_levels: {'High': 3303, 'Medium': 1087, 'Low': 110}
- avg_ai_usage_hours: 10.35
- avg_manual_hours: 22.37
- avg_burnout_score: 8.35
- avg_productivity_score: 64.95
- avg_work_life_balance: 4.72


In [5]:
# Define the style instructions
STYLE_INSTRUCTIONS = {
    "executive": 
    """
    Respond in an Executive Summary style.
    - Use plain, concise language.
    - Emphasize the main takeaway first.
    - Mention practical workplace implications.
    - Avoid technical jargon unless necessary.
    - Keep the answer brief and decision-oriented.
    """,
    "analytical": 
    """
    Respond in an Analytical Explanation style.
    - Explain the main patterns clearly.
    - Compare relevant groups or variables where useful.
    - Balance clarity and detail.
    - Keep the answer grounded in the dataset.
    - Do not overstate conclusions.
    """,
    "technical": """
    Respond in a Technical Interpretation style.
    - Refer explicitly to dataset variables where relevant.
    - Emphasize associations, not causality.
    - Mention limitations or ambiguity when appropriate.
    - Use more precise analytical language.
    - Keep the answer dataset-grounded and method-aware.
    """
}


The following questions are set up to be run by the model for each response style under consideration.

In [6]:
# Define the evaluation questions
questions = [
    "Which job roles appear to have the highest burnout risk?",
    "How does AI tool usage relate to burnout risk in this dataset?",
    "Show employees with high burnout risk and low productivity, and summarize what they have in common.",
    "What workplace factors seem most associated with burnout in this dataset?"
]

In [7]:
# Initialize the model

load_dotenv()
anthropic_key = os.getenv("ANTHROPIC_API_KEY")

if not anthropic_key:
    raise ValueError("ANTHROPIC_API_KEY is not set in your environment.")

chat_model = ChatAnthropic(model="claude-sonnet-4-0")

In [8]:
# Build a reusable prompt function

def build_prompt(question: str, style_key: str, data_description: str, dataset_snapshot: str) -> str:
    return f"""
        You are an AI assistant for a workplace burnout analytics dashboard.
        
        {STYLE_INSTRUCTIONS[style_key]}
        
        Context about the dataset:
        {data_description}
        
        {dataset_snapshot}
        
        User question:
        {question}
        
        Instructions:
        - Answer using only the dataset context provided.
        - If the question suggests causality, reframe it as association.
        - If a precise subgroup is requested, describe how you would identify it from the dataset fields.
        - Be truthful about uncertainty.
    """

In [ ]:
# Run the experiment
def ask_model(question: str, style_key: str) -> str:
    """Send a question to the LLM using the specified response style and return the generated answer."""
    prompt = build_prompt(
        question=question,
        style_key=style_key,
        data_description=data_description,
        dataset_snapshot=dataset_snapshot,
    )
    response = chat_model.chat(prompt)
    return str(response)

def clean_response(text: str) -> str:
    """Convert model output into compact plain text."""
    if not isinstance(text, str):
        return ""
    
    # Remove markdown emphasis markers
    text = re.sub(r"[*_`#>-]+", "", text)

    # Collapse multiple spaces/newlines
    text = re.sub(r"\s+", " ", text).strip()

    return text


records = []

for question in questions:
    for style_key in ["executive", "analytical", "technical"]:
        print(f"Running: {style_key} | {question}")
        answer = ask_model(question, style_key)
        records.append(
            {
                "question": question,
                "style": style_key,
                "response": clean_response(answer),
            }
        )

In [10]:
# Convert records to dataframe 
experiment_df = pd.DataFrame(records)
experiment_df.head()

,question,style,response
0,Which job roles appear to have the highest bur...,executive,Executive Summary: Burnout Risk by Job Role Ke...
1,Which job roles appear to have the highest bur...,analytical,Burnout Risk Patterns by Job Role Overall Cont...
2,Which job roles appear to have the highest bur...,technical,Technical Analysis: Burnout Risk Distribution ...
3,How does AI tool usage relate to burnout risk ...,executive,Executive Summary: AI Tool Usage and Burnout R...
4,How does AI tool usage relate to burnout risk ...,analytical,AI Tool Usage and Burnout Risk Associations Da...


In [11]:
# Verify that md markers have been removed
print(experiment_df.iloc[1, :])

question    Which job roles appear to have the highest bur...
style                                              analytical
response    Burnout Risk Patterns by Job Role Overall Cont...
Name: 1, dtype: str


In [12]:
# Save the experiment results
output_dir = Path("outputs")
output_dir.mkdir(parents=True, exist_ok=True)

experiment_path = output_dir / "querychat_response_style_experiment_results.csv"
experiment_df.to_csv(experiment_path, index=False)

experiment_path

WindowsPath('outputs/querychat_response_style_experiment_results.csv')

### Evaluation

Based on relevance to the dataset, clarity for dashboard users, actionability of insights, appropriateness for different user audiences, and faithfulness to the data without unsupported claims, we qualitatively evaluate the responses. Together, these criteria assess whether a response style genuinely improves the interpretability of AI‑generated explanations.

In [13]:
# Create evaluation table from experiment results
evaluation_template = experiment_df.copy()

# Add explicit scoring columns
# Scale: 1 = poor, 3 = acceptable, 5 = strong
evaluation_template["relevance_score"] = pd.NA
evaluation_template["clarity_score"] = pd.NA
evaluation_template["actionability_score"] = pd.NA
evaluation_template["audience_fit_score"] = pd.NA
evaluation_template["faithfulness_score"] = pd.NA

# Overall summary score for easier comparison later
evaluation_template["overall_score"] = pd.NA

# Short qualitative notes for what worked/did not work
evaluation_template["notes"] = ""

# Final recommendation flag per row
evaluation_template["recommended_for"] = ""

# Save blank evaluation template
evaluation_path = output_dir / "querychat_response_style_scores.csv"
evaluation_template.to_csv(evaluation_path, index=False)

print("Table 2: Response Style Evaluation Template\n")
evaluation_template.head()

Table 2: Response Style Evaluation Template



,question,style,response,relevance_score,clarity_score,actionability_score,audience_fit_score,faithfulness_score,overall_score,notes,recommended_for
0,Which job roles appear to have the highest bur...,executive,Executive Summary: Burnout Risk by Job Role Ke...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,,
1,Which job roles appear to have the highest bur...,analytical,Burnout Risk Patterns by Job Role Overall Cont...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,,
2,Which job roles appear to have the highest bur...,technical,Technical Analysis: Burnout Risk Distribution ...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,,
3,How does AI tool usage relate to burnout risk ...,executive,Executive Summary: AI Tool Usage and Burnout R...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,,
4,How does AI tool usage relate to burnout risk ...,analytical,AI Tool Usage and Burnout Risk Associations Da...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,,


In [14]:
evaluation_df = experiment_df.copy()

evaluation_df["relevance_score"] = pd.NA
evaluation_df["clarity_score"] = pd.NA
evaluation_df["actionability_score"] = pd.NA
evaluation_df["audience_fit_score"] = pd.NA
evaluation_df["faithfulness_score"] = pd.NA
evaluation_df["notes"] = ""

In [15]:
# Style-level scoring based on qualitative review of outputs
score_map = {
    ("Which job roles appear to have the highest burnout risk?", "executive"): {
        "relevance_score": 4,
        "clarity_score": 5,
        "actionability_score": 3,
        "audience_fit_score": 4,
        "faithfulness_score": 3,
        "notes": "Clear and accessible, but somewhat general."
    },
    ("Which job roles appear to have the highest burnout risk?", "analytical"): {
        "relevance_score": 5,
        "clarity_score": 4,
        "actionability_score": 4,
        "audience_fit_score": 5,
        "faithfulness_score": 4,
        "notes": "Balanced and interpretable, with better grounding in role comparisons."
    },
    ("Which job roles appear to have the highest burnout risk?", "technical"): {
        "relevance_score": 5,
        "clarity_score": 3,
        "actionability_score": 3,
        "audience_fit_score": 3,
        "faithfulness_score": 5,
        "notes": "Most precise, but less readable for general users."
    },

    ("How does AI tool usage relate to burnout risk in this dataset?", "executive"): {
        "relevance_score": 4,
        "clarity_score": 5,
        "actionability_score": 3,
        "audience_fit_score": 4,
        "faithfulness_score": 3,
        "notes": "Readable summary, but may oversimplify the relationship."
    },
    ("How does AI tool usage relate to burnout risk in this dataset?", "analytical"): {
        "relevance_score": 5,
        "clarity_score": 4,
        "actionability_score": 4,
        "audience_fit_score": 5,
        "faithfulness_score": 4,
        "notes": "Best balance of explanation and caution."
    },
    ("How does AI tool usage relate to burnout risk in this dataset?", "technical"): {
        "relevance_score": 5,
        "clarity_score": 3,
        "actionability_score": 3,
        "audience_fit_score": 3,
        "faithfulness_score": 5,
        "notes": "Strong variable awareness and caution, but less accessible."
    },

    ("Show employees with high burnout risk and low productivity, and summarize what they have in common.", "executive"): {
        "relevance_score": 4,
        "clarity_score": 5,
        "actionability_score": 4,
        "audience_fit_score": 4,
        "faithfulness_score": 3,
        "notes": "Useful summary, though not always specific enough."
    },
    ("Show employees with high burnout risk and low productivity, and summarize what they have in common.", "analytical"): {
        "relevance_score": 5,
        "clarity_score": 4,
        "actionability_score": 5,
        "audience_fit_score": 5,
        "faithfulness_score": 4,
        "notes": "Strongest for subgroup interpretation and pattern explanation."
    },
    ("Show employees with high burnout risk and low productivity, and summarize what they have in common.", "technical"): {
        "relevance_score": 5,
        "clarity_score": 3,
        "actionability_score": 4,
        "audience_fit_score": 3,
        "faithfulness_score": 5,
        "notes": "Accurate and careful, but not as user-friendly."
    },

    ("What workplace factors seem most associated with burnout in this dataset?", "executive"): {
        "relevance_score": 4,
        "clarity_score": 5,
        "actionability_score": 3,
        "audience_fit_score": 4,
        "faithfulness_score": 3,
        "notes": "Concise but tends to stay high-level."
    },
    ("What workplace factors seem most associated with burnout in this dataset?", "analytical"): {
        "relevance_score": 5,
        "clarity_score": 4,
        "actionability_score": 4,
        "audience_fit_score": 5,
        "faithfulness_score": 4,
        "notes": "Clear explanation of likely associated factors without overclaiming."
    },
    ("What workplace factors seem most associated with burnout in this dataset?", "technical"): {
        "relevance_score": 5,
        "clarity_score": 3,
        "actionability_score": 3,
        "audience_fit_score": 3,
        "faithfulness_score": 5,
        "notes": "Most rigorous, but least approachable."
    },
}

In [16]:
evaluation_df = experiment_df.copy()

for col in [
    "relevance_score",
    "clarity_score",
    "actionability_score",
    "audience_fit_score",
    "faithfulness_score",
    "notes",
]:
    evaluation_df[col] = pd.NA

for i, row in evaluation_df.iterrows():
    key = (row["question"], row["style"])
    if key in score_map:
        for col, value in score_map[key].items():
            evaluation_df.loc[i, col] = value

score_cols = [
    "relevance_score",
    "clarity_score",
    "actionability_score",
    "audience_fit_score",
    "faithfulness_score",
]

evaluation_df["overall_score"] = (
    evaluation_df[score_cols].astype(float).mean(axis=1).round(2)
)

print("Table 3: Scored Response Style Evaluation Template\n")
evaluation_df

Table 3: Scored Response Style Evaluation Template



,question,style,response,relevance_score,clarity_score,actionability_score,audience_fit_score,faithfulness_score,notes,overall_score
0,Which job roles appear to have the highest bur...,executive,Executive Summary: Burnout Risk by Job Role Ke...,4,5,3,4,3,"Clear and accessible, but somewhat general.",3.8
1,Which job roles appear to have the highest bur...,analytical,Burnout Risk Patterns by Job Role Overall Cont...,5,4,4,5,4,"Balanced and interpretable, with better ground...",4.4
2,Which job roles appear to have the highest bur...,technical,Technical Analysis: Burnout Risk Distribution ...,5,3,3,3,5,"Most precise, but less readable for general us...",3.8
3,How does AI tool usage relate to burnout risk ...,executive,Executive Summary: AI Tool Usage and Burnout R...,4,5,3,4,3,"Readable summary, but may oversimplify the rel...",3.8
4,How does AI tool usage relate to burnout risk ...,analytical,AI Tool Usage and Burnout Risk Associations Da...,5,4,4,5,4,Best balance of explanation and caution.,4.4
5,How does AI tool usage relate to burnout risk ...,technical,Technical Analysis: AI Tool Usage and Burnout ...,5,3,3,3,5,"Strong variable awareness and caution, but les...",3.8
6,Show employees with high burnout risk and low ...,executive,"Executive Summary: High Burnout, Low Productiv...",4,5,4,4,3,"Useful summary, though not always specific eno...",4.0
7,Show employees with high burnout risk and low ...,analytical,"Analysis: High Burnout Risk, Low Productivity ...",5,4,5,5,4,Strongest for subgroup interpretation and patt...,4.6
8,Show employees with high burnout risk and low ...,technical,"Technical Analysis: High Burnout Risk, Low Pro...",5,3,4,3,5,"Accurate and careful, but not as user-friendly.",4.0
9,What workplace factors seem most associated wi...,executive,Executive Summary: Key Workplace Factors Assoc...,4,5,3,4,3,Concise but tends to stay high-level.,3.8


In [17]:
final_eval_path = output_dir / "querychat_response_style_scores.csv"
evaluation_df.to_csv(final_eval_path, index=False)

final_eval_path

WindowsPath('outputs/querychat_response_style_scores.csv')

In [18]:
# Generate Style Comparison Summary
score_cols = [
    "relevance_score",
    "clarity_score",
    "actionability_score",
    "audience_fit_score",
    "faithfulness_score",
]

# Force score columns to numeric
for col in score_cols + ["overall_score"]:
    evaluation_df[col] = pd.to_numeric(evaluation_df[col], errors="coerce")

style_summary_detailed = (
    evaluation_df.groupby("style")[score_cols + ["overall_score"]]
    .mean()
    .round(2)
    .sort_values("overall_score", ascending=False)
)

print("Table 4: Detailed Evaluation Scores by Response Style\n")
style_summary_detailed

Table 4: Detailed Evaluation Scores by Response Style



,relevance_score,clarity_score,actionability_score,audience_fit_score,faithfulness_score,overall_score
style,,,,,,
analytical,5.0,4.0,4.25,5.0,4.0,4.45
executive,4.0,5.0,3.25,4.0,3.0,3.85
technical,5.0,3.0,3.25,3.0,5.0,3.85


In [19]:
style_summary_compact = (
    evaluation_df.groupby("style")[["overall_score"]]
    .mean()
    .round(2)
    .sort_values("overall_score", ascending=False)
)

print("Table 5: Overall Response Style Comparison\n")
style_summary_compact

Table 5: Overall Response Style Comparison



,overall_score
style,
analytical,4.45
executive,3.85
technical,3.85


Tables 4 and 5 summarize the average evaluation scores for each response style across the experiment questions.

## Final Decision and Discussion

In summary, we evaluated the three response styles for the QueryChat assistant: **Executive Summary**, **Analytical Explanation**, and **Technical Interpretation**, using five criteria: relevance, clarity, actionability, audience fit, and faithfulness to the dataset. 

The results showed that **Analytical Explanation achieved the highest overall score (4.45)** and performed consistently well across all metrics, providing a strong balance between interpretability, dataset grounding, and readability. Executive Summary produced the clearest responses but was sometimes too high-level, while Technical Interpretation was the most precise but less accessible to non-technical users.

Based on these results, we will implement a **Response Style dropdown** in the AI Explorer with three options: Executive Summary, Analytical Explanation *(default)*, and Technical Interpretation. This allows users to adjust the level and style of explanation while keeping Analytical Explanation as the default for balanced insights.